# Dropout e batch normalization



## Preparazione in Colab

1. Salva una copia del notebook nel tuo Drive.
2. Esegui la cella seguente e attendi «Preparazione completata».
3. Prosegui dall’alto verso il basso e completa gli esercizi.

La cella carica i dataset e i controlli dal repository del corso. Non devi installare Python né caricare manualmente i CSV. Se la sessione viene riavviata, esegui nuovamente la preparazione.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
from urllib.request import urlopen
from IPython.display import display, Markdown

course_source = 'https://raw.githubusercontent.com/prof-davidaffo/kaggle-learntools/master'
course_root = (Path("/content") if Path("/content").is_dir() else Path.home()) / "ai-lab-colab"
course_root.mkdir(parents=True, exist_ok=True)
bootstrap_path = course_root / "_bootstrap.py"
bootstrap_hash = '24c516651bf205b1c9511923ad4971fdca39a6248568d2b3aa46c198e64d1821'
if not bootstrap_path.is_file() or hashlib.sha256(bootstrap_path.read_bytes()).hexdigest() != bootstrap_hash:
    with urlopen(course_source + "/ai_lab/tools/colab_runtime.py", timeout=90) as response:
        bootstrap_data = response.read()
    if hashlib.sha256(bootstrap_data).hexdigest() != bootstrap_hash:
        raise RuntimeError("Course setup changed. Reopen the latest notebook from the course site.")
    bootstrap_path.write_bytes(bootstrap_data)
bootstrap_spec = importlib.util.spec_from_file_location("ai_lab_colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(bootstrap_spec)
bootstrap_spec.loader.exec_module(bootstrap)
bootstrap.download_runtime(course_source + "/ai_lab/colab/runtime.json", course_source, course_root)
bootstrap.setup(course_root, globals(), deep_learning=True)
display(Markdown("**Preparazione completata.** Puoi eseguire le celle della lezione."))


# Introduzione
Il deep learning comprende molti layer oltre a Dense: alcuni collegano neuroni, altri trasformano o preparano i dati. Esplora il [catalogo Keras](https://www.tensorflow.org/api_docs/python/tf/keras/layers/). Qui studiamo due layer speciali: **Dropout** e **BatchNormalization**.

# Dropout
L'overfitting può dipendere da combinazioni molto specifiche di pesi che riconoscono particolarità del campione. Il **dropout** disattiva casualmente una frazione degli ingressi di un layer a ogni passo di training, rendendo più difficile affidarsi a queste combinazioni fragili.

![Dropout casuale del 50% fra due layer](https://storage.googleapis.com/kaggle-media/learn/images/a86utxY.gif)

La rete è spinta a trovare relazioni più robuste. Un'intuizione utile è considerarlo simile a un **ensemble** di reti più piccole: combinare modelli con errori diversi può migliorare la generalizzazione, come nelle random forest.

## Aggiungerlo in Keras
**`rate`** è la frazione disattivata. Posiziona Dropout prima del layer al quale vuoi fornire ingressi con dropout:

```python
keras.Sequential([
    layers.Dropout(rate=0.3),
    layers.Dense(16),
])
```

# Batch normalization
La **batch normalization**, o *batchnorm*, può rendere l'ottimizzazione più stabile e rapida. In generale conviene mettere le feature su scale confrontabili, per esempio con [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) o [MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html). Attivazioni di grandezze molto diverse possono rendere instabili gli aggiornamenti.

BatchNormalization normalizza le attivazioni di ciascun batch usando media e deviazione standard, poi applica uno spostamento e un riscalamento con parametri apprendibili. Agisce quindi dentro la rete. Spesso riduce le epoche necessarie e può aiutare un training che si blocca; talvolta migliora anche le previsioni.

## Dove inserirla
Puoi aggiungerla dopo un layer:

```python
layers.Dense(16, activation='relu'),
layers.BatchNormalization(),
```

Oppure tra Dense e attivazione:

```python
layers.Dense(16),
layers.BatchNormalization(),
layers.Activation('relu'),
```

Come primo layer può svolgere una funzione di preprocessing adattivo. **Nota AI Lab:** non sostituisce automaticamente ogni trasformazione dei dati; durante la previsione usa statistiche accumulate nel training.

# Esempio sul vino
Aumentiamo ancora la capacità, con dropout per contenere l'overfitting e batchnorm per facilitare l'ottimizzazione. L'esempio non standardizza i dati in anticipo, per mostrare l'effetto stabilizzante di batchnorm.


In [ ]:

# Configura i grafici.
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
# Imposta i valori predefiniti di Matplotlib.
plt.rc('figure', autolayout=True)
plt.rc('axes', labelweight='bold', labelsize='large',
       titleweight='bold', titlesize=18, titlepad=10)


import pandas as pd
red_wine = pd.read_csv('../input/dl-course-data/red-wine.csv')

# Suddividi i dati in training e validazione.
df_train = red_wine.sample(frac=0.7, random_state=0)
df_valid = red_wine.drop(df_train.index)

# Separa feature e target.
X_train = df_train.drop('quality', axis=1)
X_valid = df_valid.drop('quality', axis=1)
y_train = df_train['quality']
y_valid = df_valid['quality']

Quando aggiungi dropout, potrebbe essere necessario aumentare il numero di unità nei layer **Dense**.


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Dense(1024, activation='relu', input_shape=[11]),
    layers.Dropout(0.3),
    layers.BatchNormalization(),
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.3),
    layers.BatchNormalization(),
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.3),
    layers.BatchNormalization(),
    layers.Dense(1),
])

La procedura di configurazione dell'addestramento resta la stessa.


In [ ]:
model.compile(
    optimizer='adam',
    loss='mae',
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    batch_size=256,
    epochs=100,
    verbose=0,
)


# Mostra le curve di apprendimento.
history_df = pd.DataFrame(history.history)
history_df.loc[:, ['loss', 'val_loss']].plot();

Normalmente è comunque opportuno standardizzare gli ingressi. Il fatto che l'esempio possa usare dati grezzi mostra l'utilità della batch normalization in situazioni difficili.

Ora **[migliora le previsioni su Spotify](https://prof-davidaffo.github.io/kaggle-learntools/corsi/deep_learning_intro/ex5.html)** con dropout e sperimenta batch normalization.
